In [ ]:
from pathlib import Path
import ast
import os
import sys

import pandas as pd
from graphviz import Digraph
from IPython.display import SVG, display


# ------------------------------------------------------------
# LOCATE PROJECT ROOT
# ------------------------------------------------------------

def find_project_root(start_path: Path) -> Path:
    """Find the MARS repository root from the notebook location."""

    current_path = start_path.resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "src").is_dir()
            and (candidate / "results").is_dir()
            and (candidate / "programs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the MARS-Summer-Research project root."
    )


PROJECT_ROOT = find_project_root(Path.cwd())


# ------------------------------------------------------------
# DEFINE INPUT AND OUTPUT PATHS
# ------------------------------------------------------------

PAIRED_RESULTS_PATH = (
    PROJECT_ROOT
    / "results"
    / "final"
    / "final_paired_folds.csv"
)

GP_PROGRAMS_PATH = (
    PROJECT_ROOT
    / "results"
    / "explainability_analysis"
    / "gp_programs.csv"
)

EOH_PROGRAM_INDEX_PATH = (
    PROJECT_ROOT
    / "results"
    / "explainability_analysis"
    / "eoh_program_index.csv"
)

EOH_RECURRENCE_PATH = (
    PROJECT_ROOT
    / "results"
    / "explainability_analysis"
    / "eoh_program_recurrence.csv"
)

EXPLAINABILITY_DIRECTORY = PROJECT_ROOT / "figures" / "explainability"
PERFORMANCE_DIRECTORY = PROJECT_ROOT / "figures" / "performance"
DIAGNOSTICS_DIRECTORY = PROJECT_ROOT / "figures" / "diagnostics"
SUPPORT_DIRECTORY = PROJECT_ROOT / "results" / "poster_support"

for output_directory in (
    EXPLAINABILITY_DIRECTORY,
    PERFORMANCE_DIRECTORY,
    DIAGNOSTICS_DIRECTORY,
    SUPPORT_DIRECTORY,
):
    output_directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# VERIFY REQUIRED FILES
# ------------------------------------------------------------

required_paths = {
    "Paired results": PAIRED_RESULTS_PATH,
    "GP programs": GP_PROGRAMS_PATH,
    "EOH program index": EOH_PROGRAM_INDEX_PATH,
    "EOH recurrence": EOH_RECURRENCE_PATH,
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Support directory: {SUPPORT_DIRECTORY}")
print()

for label, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:<20} {status:<8} {path}")

In [ ]:
# ------------------------------------------------------------
# LOAD FINAL RESULTS AND EXPLAINABILITY DATA
# ------------------------------------------------------------

paired_results = pd.read_csv(PAIRED_RESULTS_PATH)
gp_programs = pd.read_csv(GP_PROGRAMS_PATH)
eoh_program_index = pd.read_csv(EOH_PROGRAM_INDEX_PATH)
eoh_recurrence = pd.read_csv(EOH_RECURRENCE_PATH)


# Standardise dataset and method labels.
for dataframe in (
    paired_results,
    gp_programs,
    eoh_program_index,
    eoh_recurrence,
):
    if "dataset" in dataframe.columns:
        dataframe["dataset"] = (
            dataframe["dataset"]
            .astype(str)
            .str.upper()
            .str.strip()
        )

    if "method" in dataframe.columns:
        dataframe["method"] = (
            dataframe["method"]
            .astype(str)
            .str.lower()
            .str.strip()
        )


# ------------------------------------------------------------
# VALIDATE THE PAIRED RESULTS
# ------------------------------------------------------------

expected_datasets = {
    "FEI",
    "KSDD2",
    "MVTEC",
    "STL10",
}

expected_methods = {
    "gp_original",
    "gp_modified",
    "eoh",
    "cnn",
    "mlp",
}

required_metric_columns = [
    "train_accuracy",
    "train_balanced_accuracy",
    "train_macro_f1",
    "validation_accuracy",
    "validation_balanced_accuracy",
    "validation_macro_f1",
]

missing_metric_columns = [
    column
    for column in required_metric_columns
    if column not in paired_results.columns
]

if missing_metric_columns:
    raise KeyError(
        "Missing required metric columns: "
        f"{missing_metric_columns}"
    )

if set(paired_results["dataset"]) != expected_datasets:
    raise ValueError(
        "Unexpected dataset labels: "
        f"{sorted(paired_results['dataset'].unique())}"
    )

if set(paired_results["method"]) != expected_methods:
    raise ValueError(
        "Unexpected method labels: "
        f"{sorted(paired_results['method'].unique())}"
    )

duplicate_count = paired_results.duplicated(
    subset=[
        "dataset",
        "method",
        "experiment_seed",
        "fold",
    ]
).sum()

missing_metric_values = (
    paired_results[required_metric_columns]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# DISPLAY AUDIT SUMMARY
# ------------------------------------------------------------

print("FINAL DATA AUDIT")
print("=" * 72)
print(f"Paired result rows:       {len(paired_results)}")
print(f"GP program rows:          {len(gp_programs)}")
print(f"EOH program rows:         {len(eoh_program_index)}")
print(f"EOH recurrence rows:      {len(eoh_recurrence)}")
print(f"Duplicate paired rows:    {duplicate_count}")
print()

print("Non-missing metric values")
print("-" * 72)

for column in required_metric_columns:
    non_missing = paired_results[column].notna().sum()
    print(
        f"{column:<32} "
        f"{non_missing:>3}/{len(paired_results)}"
    )

print()
print("Rows per dataset and method")
print("-" * 72)

display(
    paired_results.groupby(
        ["dataset", "method"]
    )
    .size()
    .unstack()
)

In [ ]:
# ------------------------------------------------------------
# BUILD A DEFENSIBLE GP PROGRAM SHORTLIST
# ------------------------------------------------------------

gp_candidates = gp_programs.copy()

gp_candidates["validation_macro_f1"] = pd.to_numeric(
    gp_candidates["validation_macro_f1"],
    errors="coerce",
)

gp_candidates["function_call_count"] = pd.to_numeric(
    gp_candidates["function_call_count"],
    errors="coerce",
)

gp_candidates["max_parenthesis_depth"] = pd.to_numeric(
    gp_candidates["max_parenthesis_depth"],
    errors="coerce",
)


# Count exact recurrence of each program within its dataset and method.
gp_candidates["selection_count"] = (
    gp_candidates.groupby(
        ["dataset", "method", "program"]
    )["program"]
    .transform("size")
)


# Calculate the median performance for each dataset-method combination.
gp_candidates["group_median_macro_f1"] = (
    gp_candidates.groupby(
        ["dataset", "method"]
    )["validation_macro_f1"]
    .transform("median")
)


# Measure how close each program is to typical performance.
gp_candidates["distance_from_median"] = (
    gp_candidates["validation_macro_f1"]
    - gp_candidates["group_median_macro_f1"]
).abs()


# Create a shortened expression for inspecting the shortlist.
gp_candidates["program_preview"] = (
    gp_candidates["program"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 110)
)

gp_candidates["program_preview"] = (
    gp_candidates["program_preview"]
    + gp_candidates["program"]
    .astype(str)
    .str.len()
    .gt(110)
    .map({True: "...", False: ""})
)


# Prefer:
# 1. Recurrent programs
# 2. Programs close to typical validation performance
# 3. Smaller and shallower programs
gp_shortlist = (
    gp_candidates.sort_values(
        by=[
            "dataset",
            "method",
            "selection_count",
            "distance_from_median",
            "function_call_count",
            "max_parenthesis_depth",
        ],
        ascending=[
            True,
            True,
            False,
            True,
            True,
            True,
        ],
    )
    .groupby(
        ["dataset", "method"],
        as_index=False,
    )
    .head(3)
    .copy()
)


# Give each shortlist entry a stable identifier.
gp_shortlist["candidate_id"] = (
    gp_shortlist["dataset"]
    + "_"
    + gp_shortlist["method"]
    + "_s"
    + gp_shortlist["experiment_seed"].astype(str)
    + "_f"
    + gp_shortlist["fold"].astype(str)
)


shortlist_columns = [
    "candidate_id",
    "dataset",
    "method",
    "experiment_seed",
    "fold",
    "validation_macro_f1",
    "group_median_macro_f1",
    "distance_from_median",
    "selection_count",
    "function_call_count",
    "max_parenthesis_depth",
    "unique_operator_count",
    "program_preview",
]

print("GP REPRESENTATIVE-PROGRAM SHORTLIST")
print("=" * 90)
print(
    "Selection prioritises recurrence, typical performance, "
    "and visual readability."
)
print()

display(
    gp_shortlist[shortlist_columns]
    .sort_values(
        ["dataset", "method", "distance_from_median"]
    )
    .reset_index(drop=True)
)

In [ ]:
# ------------------------------------------------------------
# SELECT THE TWO REPRESENTATIVE GP PROGRAMS
# ------------------------------------------------------------

selected_gp_candidate_ids = {
    "gp_original": "STL10_gp_original_s43_f2",
    "gp_modified": "STL10_gp_modified_s43_f2",
}


def select_gp_candidate(
    shortlist: pd.DataFrame,
    candidate_id: str,
) -> pd.Series:
    """Retrieve exactly one shortlisted GP candidate."""

    matches = shortlist.loc[
        shortlist["candidate_id"] == candidate_id
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one match for {candidate_id}, "
            f"but found {len(matches)}."
        )

    return matches.iloc[0].copy()


selected_gp_programs = {
    method: select_gp_candidate(
        gp_shortlist,
        candidate_id,
    )
    for method, candidate_id
    in selected_gp_candidate_ids.items()
}


print("SELECTED REPRESENTATIVE GP PROGRAMS")
print("=" * 80)

for method, row in selected_gp_programs.items():
    method_label = method.replace("_", " ").title()

    print()
    print(method_label)
    print("-" * 80)
    print(f"Candidate:           {row['candidate_id']}")
    print(f"Dataset:             {row['dataset']}")
    print(f"Seed:                {row['experiment_seed']}")
    print(f"Fold:                {row['fold']}")
    print(
        "Validation macro F1: "
        f"{row['validation_macro_f1']:.4f}"
    )
    print(
        "Method median:       "
        f"{row['group_median_macro_f1']:.4f}"
    )
    print(
        "Distance from median:"
        f" {row['distance_from_median']:.4f}"
    )
    print(
        "Exact selections:    "
        f"{int(row['selection_count'])}"
    )
    print(
        "Function calls:      "
        f"{int(row['function_call_count'])}"
    )
    print(
        "Maximum depth:       "
        f"{int(row['max_parenthesis_depth'])}"
    )
    print("Program:")
    print(row["program"])

In [ ]:
# ------------------------------------------------------------
# PARSE AND RENDER FINAL GP PROGRAMS
# ------------------------------------------------------------

def normalise_program_text(program_text: str) -> str:
    """Prepare a saved GP expression for AST parsing."""

    text = str(program_text).strip()

    if (
        len(text) >= 2
        and text[0] == text[-1]
        and text[0] in {"'", '"'}
    ):
        text = text[1:-1].strip()

    return text


def parse_gp_program(program_text: str):
    """Parse a function-style GP expression."""

    cleaned_program = normalise_program_text(
        program_text
    )

    try:
        return ast.parse(
            cleaned_program,
            mode="eval",
        ).body

    except SyntaxError as error:
        raise ValueError(
            "Could not parse this GP program:\n"
            f"{cleaned_program}"
        ) from error


# Poster-compatible colours.
COLOURS = {
    "navy": "#283D53",
    "dark_navy": "#102F50",
    "white": "#FFFFFF",
    "light_text": "#E8EDF2",
    "green": "#A8C51B",
    "light_green": "#DCE8A1",
    "purple": "#8E72E8",
    "light_purple": "#D7CDF8",
    "blue": "#6CA6D9",
    "light_blue": "#DCEBFA",
    "orange": "#F2A23A",
    "light_orange": "#FCE4BD",
    "pink": "#D985C5",
    "light_pink": "#F2D7EC",
}


FEATURE_HINTS = (
    "SIFT",
    "HOG",
    "LBP",
    "ULBP",
    "RIF",
    "HIST",
    "GABOR",
    "ORB",
)

PROCESSING_HINTS = (
    "LOG",
    "GAU",
    "GAUD",
    "SOBEL",
    "LAP",
    "DIF",
    "MIN",
    "MAX",
    "MED",
    "MEAN",
)

REGION_HINTS = (
    "REGION",
    "RECT",
    "PATCH",
)

CONCATENATION_HINTS = (
    "FC",
    "CONCAT",
    "COMBINE",
)


def get_gp_node_style(
    label: str,
    *,
    is_function: bool,
):
    """Choose a node style from its operator role."""

    upper_label = label.upper()

    if not is_function:
        if upper_label in {
            "IMAGE",
            "IMG",
            "ARG0",
            "X",
        }:
            return {
                "shape": "ellipse",
                "style": "filled",
                "fillcolor": COLOURS["green"],
                "color": COLOURS["green"],
                "fontcolor": COLOURS["dark_navy"],
                "penwidth": "1.8",
            }

        return {
            "shape": "circle",
            "style": "filled",
            "fillcolor": COLOURS["light_pink"],
            "color": COLOURS["pink"],
            "fontcolor": COLOURS["dark_navy"],
            "fontsize": "10",
            "width": "0.38",
            "height": "0.38",
            "fixedsize": "false",
        }

    if any(
        upper_label.startswith(hint)
        for hint in CONCATENATION_HINTS
    ):
        return {
            "shape": "box",
            "style": "rounded,filled",
            "fillcolor": COLOURS["purple"],
            "color": COLOURS["light_purple"],
            "fontcolor": COLOURS["white"],
            "penwidth": "2.2",
        }

    if any(
        hint in upper_label
        for hint in REGION_HINTS
    ):
        return {
            "shape": "box",
            "style": "rounded,filled",
            "fillcolor": COLOURS["light_orange"],
            "color": COLOURS["orange"],
            "fontcolor": COLOURS["dark_navy"],
            "penwidth": "1.7",
        }

    if any(
        hint in upper_label
        for hint in FEATURE_HINTS
    ):
        return {
            "shape": "box",
            "style": "rounded,filled",
            "fillcolor": COLOURS["light_green"],
            "color": COLOURS["green"],
            "fontcolor": COLOURS["dark_navy"],
            "penwidth": "1.7",
        }

    if any(
        hint in upper_label
        for hint in PROCESSING_HINTS
    ):
        return {
            "shape": "box",
            "style": "rounded,filled",
            "fillcolor": COLOURS["light_blue"],
            "color": COLOURS["blue"],
            "fontcolor": COLOURS["dark_navy"],
            "penwidth": "1.7",
        }

    return {
        "shape": "box",
        "style": "rounded,filled",
        "fillcolor": COLOURS["light_purple"],
        "color": COLOURS["purple"],
        "fontcolor": COLOURS["dark_navy"],
        "penwidth": "1.7",
    }


class GPTreeRenderer:
    """Convert a parsed GP expression into a Graphviz tree."""

    def __init__(self, graph):
        self.graph = graph
        self.node_counter = 0

    def new_node_id(self) -> str:
        node_id = f"node_{self.node_counter}"
        self.node_counter += 1
        return node_id

    @staticmethod
    def function_name(node) -> str:
        if isinstance(node, ast.Name):
            return node.id

        if isinstance(node, ast.Attribute):
            return ast.unparse(node)

        return ast.unparse(node)

    @staticmethod
    def terminal_label(node) -> str:
        if isinstance(node, ast.Name):
            return node.id

        if isinstance(node, ast.Constant):
            return str(node.value)

        return ast.unparse(node)

    def add_ast_node(
        self,
        ast_node,
        *,
        is_root: bool = False,
    ) -> str:
        graph_node_id = self.new_node_id()

        if isinstance(ast_node, ast.Call):
            label = self.function_name(ast_node.func)

            self.graph.node(
                graph_node_id,
                label,
                **get_gp_node_style(
                    label,
                    is_function=True,
                ),
            )

            for argument_index, argument in enumerate(
                ast_node.args,
                start=1,
            ):
                child_id = self.add_ast_node(argument)

                edge_attributes = {}

                # Label only the top-level feature branches.
                if is_root:
                    edge_attributes["label"] = (
                        f"Feature {argument_index}"
                    )

                self.graph.edge(
                    graph_node_id,
                    child_id,
                    **edge_attributes,
                )

            return graph_node_id

        label = self.terminal_label(ast_node)

        self.graph.node(
            graph_node_id,
            label,
            **get_gp_node_style(
                label,
                is_function=False,
            ),
        )

        return graph_node_id


def render_gp_tree(
    row: pd.Series,
    output_stem: str,
):
    """Render one selected GP program in poster style."""

    program_tree = parse_gp_program(
        row["program"]
    )

    graph = Digraph(
        name=output_stem,
        format="svg",
    )

    graph.attr(
        rankdir="TB",
        bgcolor="transparent",
        pad="0.15",
        margin="0",
        nodesep="0.28",
        ranksep="0.42",
        splines="polyline",
    )

    graph.attr(
        "node",
        fontname="Arial",
        fontsize="13",
        margin="0.14,0.09",
    )

    graph.attr(
        "edge",
        fontname="Arial",
        fontsize="9",
        color=COLOURS["light_text"],
        fontcolor=COLOURS["light_text"],
        penwidth="1.5",
        arrowsize="0.65",
    )

    renderer = GPTreeRenderer(graph)
    renderer.add_ast_node(
        program_tree,
        is_root=True,
    )

    saved_paths = {}

    for output_format in (
        "svg",
        "png",
        "pdf",
    ):
        graph.format = output_format

        saved_path = graph.render(
            filename=str(
                EXPLAINABILITY_DIRECTORY / output_stem
            ),
            cleanup=True,
        )

        saved_paths[output_format] = Path(
            saved_path
        )

    return saved_paths

In [ ]:
# ------------------------------------------------------------
# GENERATE AND DISPLAY THE TWO GP TREES
# ------------------------------------------------------------

from IPython.display import HTML


gp_tree_outputs = {}

for method, row in selected_gp_programs.items():
    output_stem = (
        f"{row['dataset'].lower()}_"
        f"{method}_representative_tree"
    )

    saved_paths = render_gp_tree(
        row=row,
        output_stem=output_stem,
    )

    gp_tree_outputs[method] = saved_paths

    method_label = method.replace(
        "_",
        " ",
    ).title()

    print()
    print(method_label)
    print("-" * 72)

    for output_format, path in saved_paths.items():
        print(
            f"{output_format.upper():<5} "
            f"{path}"
        )

    svg_content = (
        saved_paths["svg"]
        .read_text(encoding="utf-8")
    )

    display(
        HTML(
            f"""
            <div style="
                background: {COLOURS['navy']};
                padding: 24px;
                margin: 12px 0 28px 0;
                border-radius: 4px;
                text-align: center;
            ">
                <h3 style="
                    color: white;
                    font-family: Arial;
                    margin: 0 0 12px 0;
                ">
                    {method_label}
                </h3>
                {svg_content}
            </div>
            """
        )
    )

In [ ]:
# ------------------------------------------------------------
# SELECT A REPRESENTATIVE FINAL EOH PROGRAM
# ------------------------------------------------------------

eoh_result_metrics = paired_results.loc[
    paired_results["method"] == "eoh",
    [
        "dataset",
        "method",
        "experiment_seed",
        "fold",
        "validation_macro_f1",
    ],
].copy()


eoh_candidates = eoh_program_index.merge(
    eoh_result_metrics,
    on=[
        "dataset",
        "method",
        "experiment_seed",
        "fold",
    ],
    how="left",
    validate="one_to_one",
)


eoh_candidates = eoh_candidates.merge(
    eoh_recurrence[
        [
            "dataset",
            "canonical_hash",
            "selection_count",
        ]
    ],
    on=[
        "dataset",
        "canonical_hash",
    ],
    how="left",
    validate="many_to_one",
)


eoh_candidates["dataset_median_macro_f1"] = (
    eoh_candidates.groupby(
        "dataset"
    )["validation_macro_f1"]
    .transform("median")
)


eoh_candidates["distance_from_median"] = (
    eoh_candidates["validation_macro_f1"]
    - eoh_candidates["dataset_median_macro_f1"]
).abs()


# STL-10 is selected because its final EOH programs contain
# explicit colour, intensity, spatial and edge information.
stl10_eoh_candidates = eoh_candidates.loc[
    eoh_candidates["dataset"] == "STL10"
].copy()


# Prefer the most recurrent program, then the result closest
# to typical STL-10 EOH performance.
selected_eoh_program = (
    stl10_eoh_candidates.sort_values(
        by=[
            "selection_count",
            "distance_from_median",
            "function_call_count",
            "experiment_seed",
            "fold",
        ],
        ascending=[
            False,
            True,
            True,
            True,
            True,
        ],
    )
    .iloc[0]
    .copy()
)


# Resolve the stored Windows-style relative path.
relative_program_path = Path(
    str(selected_eoh_program["path"])
    .replace("\\", os.sep)
)

selected_eoh_program_path = (
    PROJECT_ROOT / relative_program_path
)

if not selected_eoh_program_path.exists():
    raise FileNotFoundError(
        "Selected EOH program was not found:\n"
        f"{selected_eoh_program_path}"
    )


selected_eoh_source = (
    selected_eoh_program_path.read_text(
        encoding="utf-8"
    )
)


print("SELECTED REPRESENTATIVE EOH PROGRAM")
print("=" * 80)
print(f"Dataset:              {selected_eoh_program['dataset']}")
print(f"Seed:                 {selected_eoh_program['experiment_seed']}")
print(f"Fold:                 {selected_eoh_program['fold']}")
print(f"Program hash:         {selected_eoh_program['canonical_hash']}")
print(f"Selections:           {int(selected_eoh_program['selection_count'])}/10")
print(
    "Validation macro F1: "
    f"{selected_eoh_program['validation_macro_f1']:.4f}"
)
print(
    "Dataset median:      "
    f"{selected_eoh_program['dataset_median_macro_f1']:.4f}"
)
print(
    "Returned features:   "
    f"{int(selected_eoh_program['returned_feature_count'])}"
)
print(f"Program path:         {selected_eoh_program_path}")
print()
print("PROGRAM SOURCE")
print("-" * 80)
print(selected_eoh_source)

In [ ]:
# ------------------------------------------------------------
# PARSE THE EIGHT RETURNED EOH FEATURES
# ------------------------------------------------------------

eoh_ast = ast.parse(selected_eoh_source)

extract_features_function = next(
    (
        node
        for node in eoh_ast.body
        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
        and node.name == "extract_features"
    ),
    None,
)

if extract_features_function is None:
    raise ValueError(
        "Could not find extract_features in the "
        "selected EOH program."
    )


return_nodes = [
    node
    for node in ast.walk(
        extract_features_function
    )
    if isinstance(node, ast.Return)
]

if len(return_nodes) != 1:
    raise ValueError(
        "Expected exactly one return statement, "
        f"but found {len(return_nodes)}."
    )


return_value = return_nodes[0].value

if not (
    isinstance(return_value, ast.Call)
    and return_value.args
):
    raise ValueError(
        "The return statement is not a recognised "
        "NumPy array construction."
    )


feature_container = return_value.args[0]

if not isinstance(
    feature_container,
    (
        ast.List,
        ast.Tuple,
    ),
):
    raise ValueError(
        "Could not locate the returned feature list."
    )


eoh_feature_expressions = [
    ast.unparse(feature_node)
    for feature_node
    in feature_container.elts
]


# Human-readable labels are explicitly mapped to the
# verified source expressions.
feature_metadata = {
    "np.mean(red)": {
        "label": "Mean red channel",
        "family": "Colour",
        "meaning": "Average red intensity",
    },
    "np.mean(green)": {
        "label": "Mean green channel",
        "family": "Colour",
        "meaning": "Average green intensity",
    },
    "np.mean(blue)": {
        "label": "Mean blue channel",
        "family": "Colour",
        "meaning": "Average blue intensity",
    },
    "np.std(intensity)": {
        "label": "Intensity variation",
        "family": "Intensity",
        "meaning": "Global contrast",
    },
    "np.mean(centre) - np.mean(intensity)": {
        "label": "Centre-global contrast",
        "family": "Spatial",
        "meaning": "Central brightness relative to image",
    },
    "np.mean(np.abs(dx))": {
        "label": "Horizontal edge strength",
        "family": "Gradient",
        "meaning": "Average horizontal intensity change",
    },
    "np.mean(np.abs(dy))": {
        "label": "Vertical edge strength",
        "family": "Gradient",
        "meaning": "Average vertical intensity change",
    },
    "np.mean(colour_range)": {
        "label": "Colour range",
        "family": "Colour",
        "meaning": "Average within-pixel channel spread",
    },
}


unrecognised_features = [
    expression
    for expression in eoh_feature_expressions
    if expression not in feature_metadata
]

if unrecognised_features:
    raise ValueError(
        "Unrecognised returned feature expressions:\n"
        + "\n".join(unrecognised_features)
    )


eoh_feature_table = pd.DataFrame(
    [
        {
            "feature_number": index,
            "expression": expression,
            **feature_metadata[expression],
        }
        for index, expression
        in enumerate(
            eoh_feature_expressions,
            start=1,
        )
    ]
)


if len(eoh_feature_table) != 8:
    raise ValueError(
        "Expected eight EOH features, "
        f"but extracted {len(eoh_feature_table)}."
    )


print("VERIFIED EOH FEATURE VECTOR")
print("=" * 80)
print(
    f"Extracted {len(eoh_feature_table)} "
    "features directly from the selected program."
)
print()

display(eoh_feature_table)

In [ ]:
# ------------------------------------------------------------
# RENDER THE FINAL EOH EIGHT-FEATURE PIPELINE
# ------------------------------------------------------------

def render_eoh_feature_pipeline(
    feature_table: pd.DataFrame,
    selected_program: pd.Series,
    output_stem: str,
):
    """Render the selected EOH program as an explicit pipeline."""

    grouped_features = {
        family: group.copy()
        for family, group in feature_table.groupby(
            "family",
            sort=False,
        )
    }

    family_styles = {
        "Colour": {
            "fillcolor": COLOURS["light_purple"],
            "color": COLOURS["purple"],
        },
        "Intensity": {
            "fillcolor": COLOURS["light_green"],
            "color": COLOURS["green"],
        },
        "Spatial": {
            "fillcolor": COLOURS["light_orange"],
            "color": COLOURS["orange"],
        },
        "Gradient": {
            "fillcolor": COLOURS["light_blue"],
            "color": COLOURS["blue"],
        },
    }

    graph = Digraph(
        name=output_stem,
        format="svg",
    )

    graph.attr(
        rankdir="TB",
        bgcolor="transparent",
        pad="0.15",
        margin="0",
        nodesep="0.28",
        ranksep="0.42",
        splines="polyline",
    )

    graph.attr(
        "node",
        shape="box",
        style="rounded,filled",
        fontname="Arial",
        fontsize="12",
        margin="0.18,0.12",
        fontcolor=COLOURS["dark_navy"],
        penwidth="1.8",
    )

    graph.attr(
        "edge",
        color=COLOURS["light_text"],
        penwidth="1.5",
        arrowsize="0.65",
    )

    # Input image
    graph.node(
        "input",
        "96 x 96 RGB image",
        shape="ellipse",
        style="filled",
        fillcolor=COLOURS["green"],
        color=COLOURS["green"],
        fontcolor=COLOURS["dark_navy"],
        penwidth="2.0",
    )

    # Selected EOH program
    graph.node(
        "program",
        (
            "EOH-generated Python program\n"
            "8 explicit features\n"
            f"Recovered in "
            f"{int(selected_program['selection_count'])}/10 folds"
        ),
        shape="box",
        style="rounded,filled",
        fillcolor=COLOURS["purple"],
        color=COLOURS["light_purple"],
        fontcolor=COLOURS["white"],
        penwidth="2.2",
    )

    graph.edge(
        "input",
        "program",
    )

    family_node_ids = []

    # Create one box for each feature family.
    for family, group in grouped_features.items():
        family_node_id = (
            "family_"
            + family.lower()
        )

        family_node_ids.append(
            family_node_id
        )

        feature_lines = "\n".join(
            f"{int(row.feature_number)}. {row.label}"
            for row in group.itertuples()
        )

        node_label = (
            f"{family.upper()} "
            f"({len(group)})\n"
            f"{feature_lines}"
        )

        graph.node(
            family_node_id,
            node_label,
            shape="box",
            style="rounded,filled",
            fontname="Arial",
            fontsize="12",
            margin="0.18,0.12",
            fontcolor=COLOURS["dark_navy"],
            penwidth="1.8",
            **family_styles[family],
        )

        graph.edge(
            "program",
            family_node_id,
        )

    # Keep the four feature-family boxes aligned.
    with graph.subgraph(
        name="feature_family_rank"
    ) as feature_rank:
        feature_rank.attr(rank="same")

        for node_id in family_node_ids:
            feature_rank.node(node_id)

    # Explicit feature vector
    graph.node(
        "feature_vector",
        (
            "Explicit feature vector\n"
            "[f1, f2, ..., f8]"
        ),
        shape="box",
        style="rounded,filled",
        fillcolor=COLOURS["light_purple"],
        color=COLOURS["purple"],
        fontcolor=COLOURS["dark_navy"],
        penwidth="2.0",
    )

    for node_id in family_node_ids:
        graph.edge(
            node_id,
            "feature_vector",
        )

    # Classifier
    graph.node(
        "classifier",
        "Linear SVM",
        shape="box",
        style="rounded,filled",
        fillcolor=COLOURS["light_green"],
        color=COLOURS["green"],
        fontcolor=COLOURS["dark_navy"],
        penwidth="2.0",
    )

    # Output prediction
    graph.node(
        "prediction",
        "Predicted class",
        shape="ellipse",
        style="filled",
        fillcolor=COLOURS["green"],
        color=COLOURS["green"],
        fontcolor=COLOURS["dark_navy"],
        penwidth="2.0",
    )

    graph.edge(
        "feature_vector",
        "classifier",
    )

    graph.edge(
        "classifier",
        "prediction",
    )

    # Export all required formats.
    saved_paths = {}

    for output_format in (
        "svg",
        "png",
        "pdf",
    ):
        graph.format = output_format

        saved_path = graph.render(
            filename=str(
                EXPLAINABILITY_DIRECTORY
                / output_stem
            ),
            cleanup=True,
        )

        saved_paths[output_format] = Path(
            saved_path
        )

    return saved_paths


# ------------------------------------------------------------
# GENERATE THE PIPELINE
# ------------------------------------------------------------

eoh_pipeline_outputs = render_eoh_feature_pipeline(
    feature_table=eoh_feature_table,
    selected_program=selected_eoh_program,
    output_stem=(
        "stl10_eoh_"
        "eight_feature_pipeline"
    ),
)


# ------------------------------------------------------------
# PRINT SAVED FILE LOCATIONS
# ------------------------------------------------------------

print("EOH EIGHT-FEATURE PIPELINE")
print("=" * 72)

for output_format, path in eoh_pipeline_outputs.items():
    print(
        f"{output_format.upper():<5} "
        f"{path}"
    )


# ------------------------------------------------------------
# DISPLAY THE SVG AGAINST THE POSTER BACKGROUND
# ------------------------------------------------------------

eoh_svg_content = (
    eoh_pipeline_outputs["svg"]
    .read_text(encoding="utf-8")
)

display(
    HTML(
        f"""
        <div style="
            background: {COLOURS['navy']};
            padding: 24px;
            margin: 12px 0 28px 0;
            border-radius: 4px;
            text-align: center;
        ">
            <h3 style="
                color: white;
                font-family: Arial;
                margin: 0 0 12px 0;
            ">
                EOH: Explicit Eight-Feature Program
            </h3>
            {eoh_svg_content}
        </div>
        """
    )
)

In [ ]:
# ------------------------------------------------------------
# SUMMARISE TRAINING AND VALIDATION PERFORMANCE
# ------------------------------------------------------------

dataset_order = [
    "FEI",
    "KSDD2",
    "MVTEC",
    "STL10",
]

method_order = [
    "gp_original",
    "gp_modified",
    "eoh",
    "cnn",
    "mlp",
]

method_labels = {
    "gp_original": "GP Original",
    "gp_modified": "GP Modified",
    "eoh": "EOH",
    "cnn": "CNN",
    "mlp": "MLP",
}


performance_summary = (
    paired_results.groupby(
        [
            "dataset",
            "method",
        ],
        as_index=False,
    )
    .agg(
        train_macro_f1_mean=(
            "train_macro_f1",
            "mean",
        ),
        train_macro_f1_std=(
            "train_macro_f1",
            "std",
        ),
        validation_macro_f1_mean=(
            "validation_macro_f1",
            "mean",
        ),
        validation_macro_f1_std=(
            "validation_macro_f1",
            "std",
        ),
        n_observations=(
            "validation_macro_f1",
            "size",
        ),
    )
)


performance_summary["generalisation_gap"] = (
    performance_summary["train_macro_f1_mean"]
    - performance_summary[
        "validation_macro_f1_mean"
    ]
)


performance_summary["dataset"] = pd.Categorical(
    performance_summary["dataset"],
    categories=dataset_order,
    ordered=True,
)

performance_summary["method"] = pd.Categorical(
    performance_summary["method"],
    categories=method_order,
    ordered=True,
)

performance_summary = (
    performance_summary.sort_values(
        [
            "dataset",
            "method",
        ]
    )
    .reset_index(drop=True)
)

performance_summary["method_label"] = (
    performance_summary["method"]
    .astype(object)
    .map(method_labels)
)


if len(performance_summary) != 20:
    raise ValueError(
        "Expected 20 dataset-method summaries, "
        f"but found {len(performance_summary)}."
    )

if not (
    performance_summary["n_observations"] == 10
).all():
    raise ValueError(
        "Every dataset-method combination should "
        "contain 10 paired fold observations."
    )


print("TRAINING AND VALIDATION MACRO F1 SUMMARY")
print("=" * 88)
print(
    "Means and standard deviations use the 10 paired "
    "fold observations from seeds 42 and 43."
)
print()

display(
    performance_summary[
        [
            "dataset",
            "method_label",
            "train_macro_f1_mean",
            "train_macro_f1_std",
            "validation_macro_f1_mean",
            "validation_macro_f1_std",
            "generalisation_gap",
            "n_observations",
        ]
    ].style.format(
        {
            "train_macro_f1_mean": "{:.3f}",
            "train_macro_f1_std": "{:.3f}",
            "validation_macro_f1_mean": "{:.3f}",
            "validation_macro_f1_std": "{:.3f}",
            "generalisation_gap": "{:.3f}",
        }
    )
)

In [ ]:
# ------------------------------------------------------------
# TRAINING VS VALIDATION MACRO F1 DUMBBELL CHART
# ------------------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


poster_colours = {
    "background": "#283D53",
    "panel": "#24384C",
    "text": "#FFFFFF",
    "muted_text": "#C9D2DB",
    "grid": "#607386",
    "connector": "#AAB7C4",
    "training": "#9A7AEA",
    "validation": "#A8C51B",
}


fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(16, 9),
    sharex=True,
)

fig.patch.set_facecolor(
    poster_colours["background"]
)

axes = axes.flatten()


for axis, dataset in zip(
    axes,
    dataset_order,
):
    dataset_results = (
        performance_summary.loc[
            performance_summary["dataset"]
            == dataset
        ]
        .copy()
        .sort_values("method")
    )

    y_positions = list(
        range(len(dataset_results))
    )

    for y_position, row in zip(
        y_positions,
        dataset_results.itertuples(),
    ):
        train_mean = row.train_macro_f1_mean
        validation_mean = (
            row.validation_macro_f1_mean
        )

        # Connector showing the generalisation gap.
        axis.plot(
            [
                validation_mean,
                train_mean,
            ],
            [
                y_position,
                y_position,
            ],
            color=poster_colours["connector"],
            linewidth=2.2,
            zorder=1,
        )

        # Training mean and standard deviation.
        axis.errorbar(
            train_mean,
            y_position,
            xerr=row.train_macro_f1_std,
            fmt="D",
            markersize=7,
            markerfacecolor=(
                poster_colours["training"]
            ),
            markeredgecolor="#D7CDF8",
            markeredgewidth=1.3,
            ecolor=poster_colours["training"],
            elinewidth=1.5,
            capsize=3,
            zorder=3,
        )

        # Validation mean and standard deviation.
        axis.errorbar(
            validation_mean,
            y_position,
            xerr=row.validation_macro_f1_std,
            fmt="o",
            markersize=8,
            markerfacecolor=(
                poster_colours["validation"]
            ),
            markeredgecolor="#DCE8A1",
            markeredgewidth=1.3,
            ecolor=poster_colours["validation"],
            elinewidth=1.5,
            capsize=3,
            zorder=4,
        )

    axis.set_facecolor(
        poster_colours["panel"]
    )

    axis.set_title(
        dataset.replace(
            "MVTEC",
            "MVTec AD",
        ).replace(
            "STL10",
            "STL-10",
        ),
        color=poster_colours["text"],
        fontsize=19,
        fontweight="bold",
        pad=12,
    )

    axis.set_yticks(y_positions)

    axis.set_yticklabels(
        dataset_results["method_label"],
        color=poster_colours["text"],
        fontsize=13,
    )

    axis.invert_yaxis()

    axis.set_xlim(0.0, 1.05)

    axis.set_xticks(
        [
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0,
        ]
    )

    axis.tick_params(
        axis="x",
        colors=poster_colours["muted_text"],
        labelsize=11,
    )

    axis.tick_params(
        axis="y",
        length=0,
        pad=8,
    )

    axis.grid(
        axis="x",
        color=poster_colours["grid"],
        linestyle="--",
        linewidth=0.8,
        alpha=0.45,
    )

    for spine in axis.spines.values():
        spine.set_visible(False)


fig.supxlabel(
    "Macro F1",
    color=poster_colours["text"],
    fontsize=15,
    fontweight="bold",
    y=0.045,
)


legend_handles = [
    Line2D(
        [0],
        [0],
        marker="D",
        color="none",
        markerfacecolor=(
            poster_colours["training"]
        ),
        markeredgecolor="#D7CDF8",
        markersize=9,
        label="Training mean",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor=(
            poster_colours["validation"]
        ),
        markeredgecolor="#DCE8A1",
        markersize=10,
        label="Validation mean",
    ),
    Line2D(
        [0],
        [0],
        color=poster_colours["connector"],
        linewidth=2.2,
        label="Generalisation gap",
    ),
]


legend = fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.995),
    ncol=3,
    frameon=False,
    fontsize=13,
)

for legend_text in legend.get_texts():
    legend_text.set_color(
        poster_colours["text"]
    )


fig.text(
    0.5,
    0.012,
    (
        "Mean +/- SD across 10 paired fold observations "
        "(five folds; seeds 42 and 43)."
    ),
    ha="center",
    color=poster_colours["muted_text"],
    fontsize=11,
)


fig.subplots_adjust(
    left=0.15,
    right=0.98,
    top=0.90,
    bottom=0.10,
    wspace=0.24,
    hspace=0.30,
)


performance_figure_paths = {}

for output_format in (
    "svg",
    "png",
    "pdf",
):
    output_path = (
        PERFORMANCE_DIRECTORY
        / (
            "training_vs_validation_"
            f"macro_f1.{output_format}"
        )
    )

    save_arguments = {
        "facecolor": fig.get_facecolor(),
        "bbox_inches": "tight",
    }

    if output_format == "png":
        save_arguments["dpi"] = 300

    fig.savefig(
        output_path,
        **save_arguments,
    )

    performance_figure_paths[
        output_format
    ] = output_path


print("TRAINING VS VALIDATION FIGURE")
print("=" * 72)

for output_format, path in (
    performance_figure_paths.items()
):
    print(
        f"{output_format.upper():<5} "
        f"{path}"
    )

plt.show()

In [ ]:
# ------------------------------------------------------------
# KSDD2 MLP: ACCURACY CAN MASK CLASS IMBALANCE
# ------------------------------------------------------------

ksdd2_mlp_results = paired_results.loc[
    (
        paired_results["dataset"] == "KSDD2"
    )
    & (
        paired_results["method"] == "mlp"
    )
].copy()


ksdd2_metric_summary = pd.DataFrame(
    {
        "metric": [
            "Accuracy",
            "Balanced accuracy",
            "Macro F1",
        ],
        "mean": [
            ksdd2_mlp_results[
                "validation_accuracy"
            ].mean(),
            ksdd2_mlp_results[
                "validation_balanced_accuracy"
            ].mean(),
            ksdd2_mlp_results[
                "validation_macro_f1"
            ].mean(),
        ],
        "std": [
            ksdd2_mlp_results[
                "validation_accuracy"
            ].std(),
            ksdd2_mlp_results[
                "validation_balanced_accuracy"
            ].std(),
            ksdd2_mlp_results[
                "validation_macro_f1"
            ].std(),
        ],
    }
)


print("KSDD2 MLP VALIDATION METRICS")
print("=" * 60)

display(
    ksdd2_metric_summary.style.format(
        {
            "mean": "{:.3f}",
            "std": "{:.3f}",
        }
    )
)


fig, axis = plt.subplots(
    figsize=(8.2, 3.8)
)

fig.patch.set_facecolor(
    poster_colours["background"]
)

axis.set_facecolor(
    poster_colours["panel"]
)


bar_colours = [
    poster_colours["validation"],
    poster_colours["training"],
    "#F2A23A",
]


bars = axis.barh(
    ksdd2_metric_summary["metric"],
    ksdd2_metric_summary["mean"],
    xerr=ksdd2_metric_summary["std"],
    color=bar_colours,
    edgecolor=COLOURS["light_text"],
    linewidth=1.2,
    capsize=4,
)


axis.set_xlim(0.0, 1.0)

axis.set_xlabel(
    "Validation score",
    color=poster_colours["text"],
    fontsize=13,
    fontweight="bold",
)

axis.set_title(
    "KSDD2 MLP: accuracy masks weak class-balanced performance",
    color=poster_colours["text"],
    fontsize=16,
    fontweight="bold",
    pad=12,
)

axis.tick_params(
    axis="x",
    colors=poster_colours["muted_text"],
    labelsize=11,
)

axis.tick_params(
    axis="y",
    colors=poster_colours["text"],
    labelsize=12,
    length=0,
    pad=8,
)

axis.grid(
    axis="x",
    color=poster_colours["grid"],
    linestyle="--",
    linewidth=0.8,
    alpha=0.45,
)

axis.set_axisbelow(True)

for spine in axis.spines.values():
    spine.set_visible(False)


for bar, value in zip(
    bars,
    ksdd2_metric_summary["mean"],
):
    axis.text(
        min(value + 0.025, 0.94),
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left",
        color=poster_colours["text"],
        fontsize=12,
        fontweight="bold",
    )


fig.text(
    0.5,
    0.015,
    (
        "Mean +/- SD across 10 paired validation folds "
        "(seeds 42 and 43)."
    ),
    ha="center",
    color=poster_colours["muted_text"],
    fontsize=10,
)


fig.subplots_adjust(
    left=0.25,
    right=0.97,
    top=0.80,
    bottom=0.23,
)


ksdd2_inset_paths = {}

for output_format in (
    "svg",
    "png",
    "pdf",
):
    output_path = (
        DIAGNOSTICS_DIRECTORY
        / (
            "ksdd2_mlp_metric_"
            f"comparison.{output_format}"
        )
    )

    save_arguments = {
        "facecolor": fig.get_facecolor(),
        "bbox_inches": "tight",
    }

    if output_format == "png":
        save_arguments["dpi"] = 300

    fig.savefig(
        output_path,
        **save_arguments,
    )

    ksdd2_inset_paths[
        output_format
    ] = output_path


print()
print("KSDD2 SUPPORTING INSET")
print("=" * 60)

for output_format, path in (
    ksdd2_inset_paths.items()
):
    print(
        f"{output_format.upper():<5} "
        f"{path}"
    )

plt.show()

In [ ]:
# ------------------------------------------------------------
# SAVE SUPPORTING TABLES AND ASSET MANIFEST
# ------------------------------------------------------------

performance_summary_path = (
    SUPPORT_DIRECTORY
    / "performance_summary.csv"
)

eoh_feature_table_path = (
    SUPPORT_DIRECTORY
    / "eoh_feature_table.csv"
)

selection_audit_path = (
    SUPPORT_DIRECTORY
    / "program_selection_audit.csv"
)

asset_manifest_path = (
    SUPPORT_DIRECTORY
    / "visualisation_manifest.csv"
)


performance_summary.to_csv(
    performance_summary_path,
    index=False,
)

eoh_feature_table.to_csv(
    eoh_feature_table_path,
    index=False,
)


selection_audit_rows = []

for method, row in selected_gp_programs.items():
    selection_audit_rows.append(
        {
            "visualisation": (
                method.replace("_", " ").title()
                + " tree"
            ),
            "dataset": row["dataset"],
            "method": method,
            "experiment_seed": int(
                row["experiment_seed"]
            ),
            "fold": int(row["fold"]),
            "validation_macro_f1": float(
                row["validation_macro_f1"]
            ),
            "group_median_macro_f1": float(
                row["group_median_macro_f1"]
            ),
            "selection_count": int(
                row["selection_count"]
            ),
            "program_identifier": (
                row["candidate_id"]
            ),
            "program_or_path": row["program"],
            "selection_reason": (
                "Representative performance, "
                "readable complexity and relevant "
                "operator composition"
            ),
        }
    )


selection_audit_rows.append(
    {
        "visualisation": (
            "EOH eight-feature pipeline"
        ),
        "dataset": selected_eoh_program[
            "dataset"
        ],
        "method": "eoh",
        "experiment_seed": int(
            selected_eoh_program[
                "experiment_seed"
            ]
        ),
        "fold": int(
            selected_eoh_program["fold"]
        ),
        "validation_macro_f1": float(
            selected_eoh_program[
                "validation_macro_f1"
            ]
        ),
        "group_median_macro_f1": float(
            selected_eoh_program[
                "dataset_median_macro_f1"
            ]
        ),
        "selection_count": int(
            selected_eoh_program[
                "selection_count"
            ]
        ),
        "program_identifier": (
            selected_eoh_program[
                "canonical_hash"
            ]
        ),
        "program_or_path": str(
            selected_eoh_program_path.relative_to(
                PROJECT_ROOT
            )
        ),
        "selection_reason": (
            "Most recurrent STL-10 EOH program; "
            "eight explicit features; performance "
            "close to dataset median"
        ),
    }
)


selection_audit = pd.DataFrame(
    selection_audit_rows
)

selection_audit.to_csv(
    selection_audit_path,
    index=False,
)


asset_records = []


def add_asset_group(
    asset_name: str,
    asset_paths: dict,
):
    for output_format, path in asset_paths.items():
        asset_records.append(
            {
                "asset": asset_name,
                "format": output_format.upper(),
                "path": str(
                    Path(path).relative_to(
                        PROJECT_ROOT
                    )
                ),
                "exists": Path(path).exists(),
                "size_bytes": (
                    Path(path).stat().st_size
                    if Path(path).exists()
                    else None
                ),
            }
        )


add_asset_group(
    "GP Original representative tree",
    gp_tree_outputs["gp_original"],
)

add_asset_group(
    "GP Modified representative tree",
    gp_tree_outputs["gp_modified"],
)

add_asset_group(
    "EOH eight-feature pipeline",
    eoh_pipeline_outputs,
)

add_asset_group(
    "Training versus validation macro F1",
    performance_figure_paths,
)

add_asset_group(
    "KSDD2 MLP metric comparison",
    ksdd2_inset_paths,
)


asset_manifest = pd.DataFrame(
    asset_records
)

asset_manifest.to_csv(
    asset_manifest_path,
    index=False,
)


if not asset_manifest["exists"].all():
    missing_assets = asset_manifest.loc[
        ~asset_manifest["exists"],
        "path",
    ].tolist()

    raise FileNotFoundError(
        "Missing generated poster assets:\n"
        + "\n".join(missing_assets)
    )


print("POSTER VISUALISATION EXPORT COMPLETE")
print("=" * 76)
print(f"Support directory: {SUPPORT_DIRECTORY}")
print()
print("Supporting tables:")
print(f"- {performance_summary_path.name}")
print(f"- {eoh_feature_table_path.name}")
print(f"- {selection_audit_path.name}")
print(f"- {asset_manifest_path.name}")
print()
print(
    f"Verified visual assets: "
    f"{len(asset_manifest)}"
)

display(asset_manifest)